# Solutions — Processing API Payloads

Exercise notebook: `07_apis/02_json_api_processing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · Unwrap response envelopes

**Working solution**

In [ ]:
def extract_items(payload, path="data"):
    node = payload
    for key in path.split("."):
        if not isinstance(node, dict) or node.get(key) is None:
            return []
        node = node[key]
    if not isinstance(node, list):
        raise TypeError(f"expected list at {path!r}, got {type(node).__name__}")
    return node

**Explanation**

Different APIs wrap results differently (`data`, `items`, `results`, `result.records`). One small helper with a path parameter handles them all.

**Complexity:** O(depth).

**Common mistakes**

- `payload['data']` → `KeyError` on empty responses that omit the key.

## Exercise 2 — 🟡 Medium · camelCase → snake_case, recursively

**Working solution**

In [ ]:
import re

_BOUNDARY_1 = re.compile(r"([A-Z]+)([A-Z][a-z])")
_BOUNDARY_2 = re.compile(r"([a-z0-9])([A-Z])")

def to_snake(name):
    name = _BOUNDARY_1.sub(r"\1_\2", name)
    name = _BOUNDARY_2.sub(r"\1_\2", name)
    return name.replace("-", "_").lower()

def snake_keys(obj):
    if isinstance(obj, dict):
        return {to_snake(k): snake_keys(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [snake_keys(v) for v in obj]
    return obj

**Explanation**

Two regex passes: split an acronym from a following capitalised word, then split lower/digit→upper boundaries. Recursion over `dict`/`list` mirrors the JSON structure.

**Complexity:** O(size of document).

**Common mistakes**

- Only converting the top level.
- One naive regex turning `HTTPStatus` into `h_t_t_p_status`.

## Exercise 3 — 🟡 Medium · Parse timestamps in every API flavour

**Working solution**

In [ ]:
from datetime import datetime, timezone

def parse_ts(value):
    if isinstance(value, bool):
        raise ValueError(f"not a timestamp: {value!r}")
    if isinstance(value, str) and value.strip().lstrip("-").replace(".", "", 1).isdigit():
        value = float(value)
    if isinstance(value, (int, float)):
        seconds = value / 1000 if abs(value) > 1e11 else value
        return datetime.fromtimestamp(seconds, tz=timezone.utc)
    if isinstance(value, str):
        text = value.strip().replace("Z", "+00:00")
        try:
            dt = datetime.fromisoformat(text)
        except ValueError:
            raise ValueError(f"not a timestamp: {value!r}") from None
        return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)
    raise ValueError(f"not a timestamp: {value!r}")

**Explanation**

Normalize everything to aware UTC at the edge of your system; convert to local time only for display. Naive datetimes compared with aware ones raise
`TypeError` — a good thing. (`fromisoformat` accepts `Z` natively from Python 3.11; the replace keeps older versions working.)

**Complexity:** O(1).

**Common mistakes**

- Mixing naive and aware datetimes.
- Treating epoch millis as seconds → dates in the year 50000.

## Exercise 4 — 🟠 Hard · Normalize and deduplicate API orders

**Working solution**

In [ ]:
from datetime import datetime, timezone

def _ts(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)

def normalize_orders(raw):
    latest = {}
    for r in raw:
        updated = _ts(r["updated_at"])
        current = latest.get(r["order_id"])
        if current is not None and current["updated_at"] > updated:
            continue
        price = None if r.get("unit_price") is None else float(r["unit_price"])
        qty = int(r["quantity"])
        latest[r["order_id"]] = {
            "order_id": r["order_id"], "customer_id": r["customer_id"], "product_id": r["product_id"],
            "quantity": qty, "unit_price": price, "amount": None if price is None else round(qty * price, 2),
            "status": str(r["status"]).upper(), "order_ts": _ts(r["order_ts"]), "updated_at": updated,
        }
    return [latest[k] for k in sorted(latest)]

**Explanation**

'Latest version wins' dedup is a dict keyed by business key holding the best row so far — O(n), one pass, no sort needed until output.
It's `ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1`, in Python.

**Complexity:** O(n + k log k).

**Common mistakes**

- Keeping the first occurrence instead of the latest version.
- `amount = qty * None` → `TypeError`.

## Challenge — 🔴 Very Hard · Reconcile API data against the warehouse

**Working solution**

In [ ]:
def _index(rows, key):
    index, dupes = {}, set()
    for r in rows:
        k = r[key]
        if k in index:
            dupes.add(k)
        index[k] = r
    return index, sorted(dupes)

def _equal(a, b, tolerance):
    if a is None or b is None:
        return a is None and b is None
    numeric = (int, float)
    if isinstance(a, numeric) and isinstance(b, numeric) and not isinstance(a, bool) and not isinstance(b, bool):
        return abs(a - b) <= tolerance
    if isinstance(a, str) and isinstance(b, str):
        return a.strip() == b.strip()
    return a == b

def reconcile(source, target, key, fields, tolerance=0.0):
    src, src_dupes = _index(source, key)
    tgt, tgt_dupes = _index(target, key)
    common = sorted(src.keys() & tgt.keys())
    mismatches, matched = [], 0
    for k in common:
        diffs = [{"key": k, "field": f, "source": src[k].get(f), "target": tgt[k].get(f)}
                 for f in fields if not _equal(src[k].get(f), tgt[k].get(f), tolerance)]
        mismatches.extend(diffs)
        matched += not diffs
    return {"source_count": len(source), "target_count": len(target), "matched": matched,
            "missing_in_target": sorted(src.keys() - tgt.keys()), "unexpected_in_target": sorted(tgt.keys() - src.keys()),
            "mismatches": mismatches, "duplicate_keys": {"source": src_dupes, "target": tgt_dupes}}

**Explanation**

Reconciliation = index both sides by key, set algebra for presence, field-by-field comparison with type-aware equality. `matched += not diffs` adds
`True`(1)/`False`(0) — handy, but only where it stays readable. This report is what you attach to a data-quality ticket.

**Complexity:** O(n + m + k·f).

**Common mistakes**

- Float equality without tolerance (`0.1 + 0.2 != 0.3`).
- Counting rows instead of distinct keys when there are duplicates.